# Notes

* Notebooks are in JSON format - you can use `json.load(file)` to load the data into a dictionary
* Dictionary format
    * Top-level keys: `dict_keys(['cells', 'metadata', 'nbformat', 'nbformat_minor'])`
    * `data["cells"]` is a list of notebook cells, which are either Markdown or code. In Markdown, there can be HTML. Should I reformat the HTML?
        * Each cell is a dictionary. The only keys you need to care about are `"cell_type"` and `"source"`.
        * `source` contains the text as a list of strings (I think each element is one line)
    * `data["metadata"]["language info"]["name"]` contains the language (e.g. Python)
    * Should you reformat everything into Markdown? Code cells - delimit with triple backticks: ```python {code}```

In [ ]:
import glob
from pymilvus import MilvusClient
from dotenv import load_dotenv
import os
import pandas as pd
import pickle
import re

load_dotenv(override=True)

files = glob.glob("Content/**/*.ipynb", recursive=True)
files

['Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/_m3.1-data-prep-text-lab-1.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Demo_LLM_Data_Prep_101.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Demo_Text_Data_Augmentation_Synthetic_Data.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Exercise_Solution_Data_Augmentation_GPT.ipynb',
 'Content/La

In [ ]:
import json

file = files[0]

with open(file, 'r') as f:
    data = json.load(f)

In [45]:
file

'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb'

In [14]:
data.keys()

dict_keys(['cells', 'metadata', 'nbformat', 'nbformat_minor'])

In [15]:
type(data["cells"])

list

In [17]:
len(data["cells"])

28

In [18]:
data["cells"][0]

{'cell_type': 'markdown',
 'metadata': {'id': 'KfqyGY5kUSy2'},
 'source': ['<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">\n',
  '<h1><center>ArXiv Data Cleaning Notebook</center></h1>']}

In [21]:
[cell["cell_type"] for cell in data["cells"]]
[f'{cell["cell_type"]}: {cell.keys()}' for cell in data["cells"]]

["markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "code: dict_keys(['cell_type', 'execution_count', 'metadata', 'outputs', 'source'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "code: dict_keys(['cell_type', 'source', 'metadata', 'execution_count', 'outputs'])",
 "markdown: dict_keys(['cell_type', 'metadata', 'source'])",
 "code: dict_keys(['cell_type', 'execution_count', 'metadata

In [33]:
for cell in data["cells"]:
    if cell["cell_type"] == "markdown":
        # print(f'{cell["source"]}\n{"-"*20}\n')
        print(type(cell["source"]))

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [32]:
for cell in data["cells"]:
    if cell["cell_type"] == "code":
        # print(f'{cell["source"]}\n{"-"*20}\n')
        print(type(cell["source"]))

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [30]:
for cell in data["cells"]:
    if cell["cell_type"] == "markdown":
        print(f'{cell["metadata"]}\n{"-"*20}\n')

{'id': 'KfqyGY5kUSy2'}
--------------------

{'id': '-tzs7zPKUSy7'}
--------------------

{'id': 'j-YdiN4u4v1C'}
--------------------

{'id': 'srWsJFlH4v1O'}
--------------------

{'id': 'dZaYV0_oliOT'}
--------------------

{'id': 'H_tESoKilqwH'}
--------------------



In [34]:
for cell in data["cells"]:
    if cell["cell_type"] == "markdown":
        # print(f'{cell["source"]}')
        for elem in cell["source"]:
            print(elem)
        print("\n")
    elif cell["cell_type"] == "code":
        print("```python")
        # print(f'{cell["source"]}')
        for elem in cell["source"]:
            print(elem)
        print("```")
        print("\n")

<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>ArXiv Data Cleaning Notebook</center></h1>


## Introduction



Welcome to the ArXiv Data Cleaning Notebook! Here, we will walk you through the process of preparing and cleaning ArXiv data using Python methods.



 - **Cleaning data with scripts**: You can clean data using scripts and EC2 accounts.



 - **Manually Cleaning data**: Dive deep into the basics and clean ArXiv datasets from scratch, offering a glimpse into the essence of use of Large Language Model.


```python
# Here is a repository we use in this lab.

!git clone https://github.com/togethercomputer/RedPajama-Data.git

%cd /content/RedPajama-Data/data_prep/arxiv
```


## Cleaning data with scripts



Follow these instructions to create the Arxiv dataset. These steps assume that you are in the data_prep/arxiv directory.


```python
# Setup

# Install the dependencies specified in arxiv_requirements.txt:

!pip install -

In [44]:
from copy import deepcopy
fname = "ipynb_sample.md"

all_cells = []
for cell in data["cells"]:

    content = deepcopy(cell["source"])
    if cell["cell_type"] == "code":
        content.insert(0, "\n```python\n")
        content.append("\n```\n")
    
    all_cells.extend(content)

writestr = "".join(all_cells)
print(writestr)

with open(fname, "w") as f:
    f.write(writestr)

<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">
<h1><center>ArXiv Data Cleaning Notebook</center></h1>## Introduction

Welcome to the ArXiv Data Cleaning Notebook! Here, we will walk you through the process of preparing and cleaning ArXiv data using Python methods.

 - **Cleaning data with scripts**: You can clean data using scripts and EC2 accounts.

 - **Manually Cleaning data**: Dive deep into the basics and clean ArXiv datasets from scratch, offering a glimpse into the essence of use of Large Language Model.
```python
# Here is a repository we use in this lab.
!git clone https://github.com/togethercomputer/RedPajama-Data.git
%cd /content/RedPajama-Data/data_prep/arxiv
```
## Cleaning data with scripts

Follow these instructions to create the Arxiv dataset. These steps assume that you are in the data_prep/arxiv directory.
```python
# Setup
# Install the dependencies specified in arxiv_requirements.txt:
!pip install -r arxiv_requirements.t

In [38]:
mylist = [1, 2, 3]

mylist.insert(0,0)

mylist.append(4)

mylist

[0, 1, 2, 3, 4]

# Get notebook kernel spec

In [70]:
import glob
from dotenv import load_dotenv
import os
import pandas as pd
import json

load_dotenv(override=True)

files = glob.glob("Content/**/*.ipynb", recursive=True)
# files

for file in files:
    with open(file, 'r') as f:
        data = json.load(f)
    
    # print(file)
    # print(data["metadata"])
    # print(data["metadata"]["kernelspec"].keys())
    # print(f'display_name: {data["metadata"]["kernelspec"].get("display_name", "none")}, name: {data["metadata"]["kernelspec"].get("name", "none")}, language: {data["metadata"]["kernelspec"].get("language", "none")}')

    print(data["metadata"]["kernelspec"].get("language", data["metadata"]["kernelspec"].get("name", data["metadata"]["kernelspec"].get("display_name", ""))))
    # display_name
    # name
    # language
    print("-"*100)

python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python3
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
----------------------------------------------------------------------------------------------------
python
--------------------

# Clean all notebooks

In [47]:
import glob
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv(override=True)

files = glob.glob("Content/**/*.ipynb", recursive=True)
files

['Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/_m3.1-data-prep-text-lab-1.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Preparation/Demo_LLM_Data_Prep_101.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Demo_Text_Data_Augmentation_Synthetic_Data.ipynb',
 'Content/Large Language Model ( LLM)/Part_3_Data_Preparation/Data_Synthesization/Exercise_Solution_Data_Augmentation_GPT.ipynb',
 'Content/La

In [71]:
import json

def convert_notebook(filename: str) -> str:
    with open(filename, 'r') as f:
        data = json.load(f)
    
    lang = data["metadata"]["kernelspec"].get("language", data["metadata"]["kernelspec"].get("name", data["metadata"]["kernelspec"].get("display_name", "")))

    all_cells = []
    for cell in data["cells"]:
        content = deepcopy(cell["source"])

        if cell["cell_type"] == "code":
            content.insert(0, f"\n```{lang}\n")
            content.append("\n```\n")
        elif cell["cell_type"] == "markdown":
            content.insert(0, "  \n")
            content.append("  \n")
        
        all_cells.extend(content)

    converted_notebook = "".join(all_cells)

    return converted_notebook

In [72]:
import re
os.makedirs("./converted_notebooks/", exist_ok=True)

for file in files:
    _, tail = os.path.split(file)
    converted_filename = re.sub(".ipynb", ".md", tail)
    
    converted_notebook = convert_notebook(file)
    with open(f"./converted_notebooks/{converted_filename}", "w") as f:
        f.write(converted_notebook)
        print(converted_filename)

Exercise_Solution_LLM_Data_Prep_Arxiv(WIP).md
Exercise_LLM_Data_Prep.md
Exercise_Solution_LLM_Data_Prep.md
_m3.1-data-prep-text-lab-1.md
Exercise_Solution_LLM_Data_Prep_for_Instruction_Tuning.md
Demo_LLM_Data_Prep_101.md
Demo_Text_Data_Augmentation_Synthetic_Data.md
Exercise_Solution_Data_Augmentation_GPT.md
Exercise_Solution_Data_Augmentation_with_BackTranslation.md
Exercise_Web_Scraping_Selenium_Linkedin.md
Exercise_Solution_Web_Scraping_BS4.md
Exercise_Solution_Web_Scraping_Selenium_Linkedin.md
Exercise_Web_Scraping_BS4.md
Demo_Web_Scraping_HTML_Xpath.md
Exercise_Solution_LLM_Annotation_Sagemaker_Groundtruth.md
Exercise_Solution_Data_Cleaning_Cleanlab_Bank_Intent.md
Exercise_Solution_Rule-based_Automatic_Data_Labeling.md
Exercise_Solution_Data_Labeling_Issue_Detection_Cleanlab_TwitterData.md
Demo_Data_Labelling_Cleanlab_Identify_Label_Issues.md
Demo_LLM-based_Labelling.md
Exercise_Data_Cleaning_Cleanlab_Bank_Intent.md
Demo_RAG_2_LangChain_Splitter.md
Demo_RAG_1_LangChain_FAISS.md
Ex